In [24]:
# Download a sample image for demonstration
!wget -O download.jpeg "https://picsum.photos/200/300"

--2026-09-25 09:01:03--  https://picsum.photos/200/300
Resolving picsum.photos (picsum.photos)... 104.26.4.30, 104.26.5.30, 172.67.74.163, ...
Connecting to picsum.photos (picsum.photos)|104.26.4.30|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://fastly.picsum.photos/id/250/200/300.jpg?hmac=igVdxs-AgITpHwPAZ80mpAfmhrGBvN_xThJlhp7vOqE [following]
--2026-09-25 09:01:03--  https://fastly.picsum.photos/id/250/200/300.jpg?hmac=igVdxs-AgITpHwPAZ80mpAfmhrGBvN_xThJlhp7vOqE
Resolving fastly.picsum.photos (fastly.picsum.photos)... 151.101.1.91, 151.101.65.91, 151.101.129.91, ...
Connecting to fastly.picsum.photos (fastly.picsum.photos)|151.101.1.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11014 (11K) [image/jpeg]
Saving to: ‘download.jpeg’

download.jpeg       100%[===================>]  10.76K  --.-KB/s    in 0s      

2026-09-25 09:01:03 (94.9 MB/s) - ‘download.jpeg’ saved [11014/11014]



In [28]:
# Install required libraries
!pip install -q transformers torch torchvision pillow

# Import libraries
import torch
from PIL import Image
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer
)

# Model name
model_name = "nlpconnect/vit-gpt2-image-captioning"

# Load pretrained model
model = VisionEncoderDecoderModel.from_pretrained(model_name)
processor = ViTImageProcessor.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Image caption generation function
def generate_caption(image_path, max_length=30, num_beams=4):

    # Open image
    image = Image.open(image_path).convert("RGB")

    # Convert image into model input
    pixel_values = processor(
        images=image,
        return_tensors="pt"
    ).pixel_values.to(device)

    # Generate caption
    output_ids = model.generate(
        pixel_values,
        max_length=max_length,
        num_beams=num_beams,
        early_stopping=True
    )

    # Convert tokens into text
    caption = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return caption


# Image file
image_path = "download.jpeg"

# Generate caption
caption = generate_caption(image_path)

print("Image Caption:", caption)

Loading weights:   0%|          | 0/445 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Image Caption: a close up of a photo of a camera 
